<img src="https://camo.githubusercontent.com/3d1152e718c0f7c8eaf876f03d500f089135acced7b2efe10d8a2292e24d3915/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f30332e77656270" width=600>

###LAYER NORMALIZATION

Definition || Normalizes each token's activations independently to Mean=0, Variance=1.

Purpose || Stable training || Better gradient flow || Faster convergence.

Need || Deep transformer stacks can cause activations to become very large or very small, leading to unstable gradients and slower learning.

Example ||

[100, 250, -300, 500]
[-500, 700, 1000, -800]

LayerNorm rescales such activations to a consistent distribution.

Intuition || Ensures every transformer block receives inputs at a similar scale regardless of previous layer outputs.

Key Benefit || Prevents exploding/shrinking activations and makes deep transformer training more stable.

_________________
###LayerNorm Learnable Parameters

Learnable Parameters || γ (gamma) = scale || β (beta) = shift

Output Formula ||

y = γx̂ + β

where x̂ = normalized input.

Purpose || Pure normalization may remove useful information || Model learns the optimal scaling and shifting after normalization.

Shapes

Input: (B,T,768)

Example:

(4,256,768)

Meaning || 4 sequences || 256 tokens/sequence || 768 features/token.

Normalization || Applied across the 768 features of each token independently.

Output Shape: (4,256,768)

Shape never changes.
______________________

###LayerNorm vs BatchNorm

BatchNorm || Normalizes across batch samples || Used in CNNs/ResNets || Depends on other samples in the batch.

LayerNorm || Normalizes across features of a single token || Used in Transformers (GPT, BERT, Llama) || Independent of batch size.

Placement in GPT Block
Input-->LayerNorm-->Multi-Head Attention-->Residual Add-->LayerNorm-->Feed Forward-->Residual Add

Key Idea || Normalize → Process → Residual Add → Normalize → Process → Residual Add.

_____________


##Scale and shift

* Note that in addition to performing the normalization by subtracting the mean and dividing by the variance, we added two trainable parameters, a scale and a shift parameter
* The initial scale (multiplying by 1) and shift (adding 0) values don't have any effect; however, scale and shift are trainable parameters that the LLM automatically adjusts during training if it is determined that doing so would improve the model's performance on its training task
* This allows the model to learn appropriate scaling and shifting that best suit the data it is processing
* Note that we also add a smaller value (eps) before computing the square root of the variance; this is to avoid division-by-zero errors if the variance is 0

In [ ]:
class LayerNorm(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5

        self.scale = nn.Parameter(
            torch.ones(emb_dim)
        )

        self.shift = nn.Parameter(
            torch.zeros(emb_dim)
        )

    def forward(self, x):

        mean = x.mean(
            dim=-1,
            keepdim=True
        )

        var = x.var(
            dim=-1,
            keepdim=True,
            unbiased=False
        )

        norm_x = (
            x - mean
        ) / torch.sqrt(
            var + self.eps
        )

        return (
            self.scale * norm_x
            + self.shift
        )

### Residual Connection

**Operation** ||

```text
attn_out = Attention(x)

x = x + attn_out
```

instead of:

```text
x = attn_out
```

**Purpose** || Preserves original information || Provides shortcut path for gradients || Improves gradient flow in deep networks.

---

### Residual + LayerNorm

**Residual** || Preserves information + helps gradients flow through many layers.

**LayerNorm** || Keeps activations stable and well-scaled.

**Together** || Enable stable training of deep transformers || Without them, GPT-2 would not train reliably.

---

### Pre-LN vs Post-LN

**Post-LN (Original Transformer)**

```text
Attention
↓
Add
↓
LayerNorm
```

**Pre-LN (GPT-2, GPT-3, Llama, Mistral)**

```text
LayerNorm
↓
Attention
↓
Add
```

**Why Pre-LN?** || Better gradient flow || More stable training for very deep transformer networks.


<img src="https://camo.githubusercontent.com/c8d3e676a7c4e7ece92848dd2064a6778672cefc1815b17fe9915a344d3961aa/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f31322e776562703f313233" width=600>

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])

    def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer
            layer_output = layer(x)
            # Check if shortcut can be applied
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x

##GELU

Formula

GELU(x)=xΦ(x)

where:

Φ(x) = Gaussian CDF

PyTorch uses a fast approximation internally.

Example

Input: [-3, -1, 0, 1, 3]

ReLU: [0, 0, 0, 1, 3]

GELU: [-0.004, -0.159, 0, 0.841, 2.996]

* Notice: Negative values are not abruptly killed.
Why GPT Uses GELU

Advantages over ReLU:

    Smoother gradients

    Better optimization

    Improved transformer performance

    Retains some information from small negative values

GELU (Gaussian Error Linear Unit)

Formula

      GELU(x)≈0.5.x.(1+tanh[root(2/π).(x+0.044715x^3)])

Purpose || Activation function used in GPT, BERT, Llama, and most modern transformers.

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

<img src="https://camo.githubusercontent.com/b5006caf8f949eeadb6f86194d0dbc68e1230c036497bf43b368b1aa5fd2c30a/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f30392e776562703f3132"  width=600>

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

GPT2-small (the 124M learnable parameters):

"emb_dim" = 768
"n_layers" = 12
"n_heads" = 12
GPT2-medium:

"emb_dim" = 1024
"n_layers" = 24
"n_heads" = 16
GPT2-large:

"emb_dim" = 1280
"n_layers" = 36
"n_heads" = 20
GPT2-XL:

"emb_dim" = 1600
"n_layers" = 48
"n_heads" = 25

<img src="https://camo.githubusercontent.com/d582825155473ca7d65fd90f47a5699283acac3268be534ef50c24db6c1d07d5/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f31332e776562703f31" width=600>
<img src="https://camo.githubusercontent.com/668de8510f9667d59162cd90c7e37d3bc35c8baf1e67b12909144069610d730b/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830345f636f6d707265737365642f31352e77656270" width=600>

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

# GPT Inference Pipeline

### Input

User Prompt || `"What is machine learning?"`

---

### 1. Tokenization

Input text → Token IDs

```text
[2061, 318, 4572, 4673, 30]
```

Shape=`(T,)` || `T` = number of tokens.

---

### 2. Context Window Check

GPT can only process a fixed number of tokens (e.g., GPT-2: 1024) || If prompt exceeds limit, oldest tokens are removed and recent tokens are kept.

---

### 3. Token Embedding

Input IDs → Lookup in `W_E:(50257,768)` || Output=`(T,768)` || Converts token IDs into dense vectors.

---

### 4. Positional Embedding

Position IDs=`[0,1,2,3,4]` || Add position embeddings from `W_P:(1024,768)`.

```text
X = TokenEmb + PosEmb
```

Output=`(T,768)` || Adds word order information.

---

# Transformer Block (Repeated N Times)

Input Shape=`(T,768)`

### 5. LayerNorm

Normalize each token independently || Shape unchanged: `(T,768)`.

---

### 6. Create Q, K, V

`Q=XWq` || `K=XWk` || `V=XWv`

`Wq,Wk,Wv:(768,768)`

Output:

```text
Q,K,V → (T,768)
```

**Q** || What information am I looking for?

**K** || What information do I contain?

**V** || What information can I provide?

---

### 7. Multi-Head Split

12 heads || `768 → 12 × 64`

Shape:

```text
(T,12,64)
```

Each head learns different relationships.

---

### 8. Attention Scores

```text
Q @ Kᵀ
```

Shape:

```text
(T,T)
```

Each token compares itself with every token in the sequence.

---

### 9. Causal Mask

Future tokens are hidden.

```text
Token i
can only attend to
tokens ≤ i
```

Purpose || Maintain autoregressive generation.

---

### 10. Softmax

Convert attention scores into probabilities.

Shape:

```text
(T,T)
```

Each row sums to 1.

---

### 11. Context Vectors

```text
AttentionWeights @ V
```

Output per head:

```text
(T,64)
```

Purpose || Gather information from important tokens.

---

### 12. Concatenate Heads

12 head outputs combined.

```text
12 × 64 = 768
```

Output:

```text
(T,768)
```

---

### 13. Output Projection

Apply:

```text
Wo:(768,768)
```

Output:

```text
(T,768)
```

Mixes information from all heads.

---

### 14. Residual Connection

```text
x = x + AttentionOutput
```

Purpose || Preserve original information + improve gradient flow.

Output:

```text
(T,768)
```

---

### 15. LayerNorm

Normalize again before FFN.

Output:

```text
(T,768)
```

---

### 16. Feed Forward Network (FFN)

Architecture:

```text
768
↓
3072
↓
768
```

Operations:

```text
Linear
↓
GELU
↓
Linear
```

Shape Flow:

```text
(T,768)
↓
(T,3072)
↓
(T,768)
```

Purpose || Per-token processing and feature transformation.

---

### 17. Residual Connection

```text
x = x + FFN_Output
```

Output:

```text
(T,768)
```

---

### 18. Repeat Transformer Blocks

GPT-2 Small → 12 Blocks

GPT-2 Large → 36 Blocks

Final Output:

```text
(T,768)
```

---

# Output Generation

### 19. Final LayerNorm

One final normalization.

Output:

```text
(T,768)
```

---

### 20. Vocabulary Projection

```text
(T,768)
@
(768,50257)
=
(T,50257)
```

Meaning || For every position, produce scores for all 50,257 vocabulary tokens.

---

### 21. Keep Last Position

Only the last token is needed for next-token prediction. As the prompt incoming is fed one word after another but we only need to predict the next word after the while prompt we keep the last position output only and dicard the prior ones

Shape:

```text
(50257,)
```

---

### 22. Softmax

Convert logits → probabilities.

Example:

```text
"It"      65%
"The"     15%
"Machine"  8%
...
```

---

### 23. Select Next Token

Methods:

* Greedy (`argmax`)
* Temperature Sampling
* Top-K Sampling
* Top-P (Nucleus) Sampling

Example:

```text
"It"
```

Token ID:

```text
632
```

---

### 24. Append Token

```text
[2061,318,4572,4673,30]
↓
[2061,318,4572,4673,30,632]
```

Prompt becomes:

```text
"What is machine learning? It"
```

---

### 25. Repeat

Updated sequence is fed back into GPT.

```text
Forward Pass
↓
Generate Next Token
↓
Append Token
↓
Repeat
```

---

### Stop Conditions

* EOS (End-of-Sequence) token generated
* Maximum token limit reached

---



# KV Cache

### Problem Without KV Cache

**Observation** || During inference, GPT generates one token at a time || Sequence grows after every step.

Example:

```text
Time flies
↓
Time flies fast
↓
Time flies fast today
```

Without KV Cache || GPT recomputes attention for **all previous tokens** at every generation step.

---

### Example

Initial Prompt:

```text
Time flies
```

Compute:

```text
Q1 K1 V1
Q2 K2 V2
```

Generate:

```text
fast
```

Sequence:

```text
Time flies fast
```

Without Cache:

```text
Q1 K1 V1
Q2 K2 V2
Q3 K3 V3
```

computed again from scratch.

Next token:

```text
Time flies fast today
```

Again recompute:

```text
Q1 K1 V1
Q2 K2 V2
Q3 K3 V3
Q4 K4 V4
```

**Waste** || Old tokens are repeatedly recomputed even though nothing about them changed.

---

### Key Observation

During inference:

* Model weights are frozen
* No training occurs
* Previously generated tokens never change

Therefore:

```text
Time
```

will always produce the same:

```text
K1
V1
```

once computed.

---

### Main Idea of KV Cache

Store previously computed:

```text
K
V
```

vectors and reuse them for future generations.

```text
Compute Once
↓
Store
↓
Reuse Forever
```

Purpose || Avoid recomputing old tokens.

---

### Why Cache Only K and V?

**Q (Query)** || Depends on the current token being processed.

Example:

```text
Current Token = fast
```

Need:

```text
Q_fast
```

which did not exist previously.

So every new token requires a new Query.

---

**K (Key) and V (Value)** || Depend only on already-generated tokens.

Example:

```text
K_time
V_time

K_flies
V_flies
```

remain unchanged and can be reused indefinitely.

---

### With KV Cache

First Pass:

```text
Time flies
```

Compute:

```text
Q1 K1 V1
Q2 K2 V2
```

Store:

```text
K1 V1
K2 V2
```

Generate:

```text
fast
```

---

Next Step:

Only compute:

```text
Q3 K3 V3
```

Reuse:

```text
K1 V1
K2 V2
```

from cache.

Attention uses:

```text
Q3
vs
[K1,K2,K3]
```

and

```text
[V1,V2,V3]
```

without recomputing old tokens.

---

### Attention Perspective

Attention:

```text
Softmax(QKᵀ)V
```

**Q** || What am I looking for?

**K** || What information do previous tokens contain?

**V** || Actual information to be mixed into the current token.

---
**Why Q Is Not Cached** || Query (`Q`) is needed only for the **current token being generated**. Every new token produces a new query because it asks a different question about the previous context. For example, when generating **"fast"**, we need `Q₃` to determine what information from previous tokens (`Time`, `flies`) is relevant. In the next step, when generating **"today"**, we need a completely different query `Q₄` because the current token and its context have changed. Since old queries (`Q₁`, `Q₂`, `Q₃`) are never used again for future predictions, caching them provides no benefit. In contrast, Keys and Values of previous tokens are repeatedly needed by every future token, so caching `K` and `V` avoids recomputing them and gives the speedup.

**Rule to Remember** ||

```text
Q → Needed once for the current token → Don't cache

K,V → Needed by all future tokens → Cache
```


### Example: Generating "fast"

Sequence:

```text
T1 = Time
T2 = flies
T3 = fast
```

Need:

```text
Q3
```

because only **fast** is asking:

```text
What previous information is useful to me?
```

Need:

```text
K1 K2 K3
```

to compute:

```text
Q3·K1
Q3·K2
Q3·K3
```

Need:

```text
V1 V2 V3
```

to create the final context vector.

---

### Most Important Insight

When generating token `T3`:

We only care about:

```text
Q3
```

We do **not** need:

```text
Q1
Q2
```

anymore.

Reason:

```text
T1 and T2 already attended
to what they needed earlier.
```

Their hidden states were already created and will never change.

---

### Attention Matrix View

For:

```text
Time flies fast
```

Attention Matrix:

```text
       T1  T2  T3

Q1      •
Q2      •   •
Q3      •   •   •
```

Without Cache:

```text
Recompute:
Q1,Q2,Q3
Entire Matrix
```

With Cache:

```text
Compute only:
Q3 row
```

Reuse:

```text
K1,K2
V1,V2
```

from cache.

---

### Key Benefit

**Without KV Cache** || Recompute all previous tokens every step || Generation becomes slower as sequence grows.

**With KV Cache** || Compute only the newest token's Q,K,V and reuse old K,V || Generation remains much faster and scales efficiently to long contexts.
_______
As sequence length increases, the benefits and downsides of a KV cache become more pronounced in the following ways:

* [Good] Computational efficiency increases: Without caching, the attention at step t must compare the new query with t previous keys, so the cumulative work scales quadratically, O(n²). With a cache, each key and value is computed once and then reused, reducing the total per-step complexity to linear, O(n).

* [Bad] Memory usage increases linearly: Each new token appends to the KV cache. For long sequences and larger LLMs, the cumulative KV cache grows larger, which can consume a significant or even prohibitive amount of (GPU) memory. As a workaround, we can truncate the KV cache, but this adds even more complexity (but again, it may well be worth it when deploying LLMs.)

In [ ]:
def forward(self, x, use_cache=False):
    b, num_tokens, d_in = x.shape

    keys_new = self.W_key(x)  # Shape: (b, num_tokens, d_out)
    values_new = self.W_value(x)
    queries = self.W_query(x)
    #...

    if use_cache:
        if self.cache_k is None:
            self.cache_k, self.cache_v = keys_new, values_new
        else:
            self.cache_k = torch.cat([self.cache_k, keys_new], dim=1)
            self.cache_v = torch.cat([self.cache_v, values_new], dim=1)
        keys, values = self.cache_k, self.cache_v
    else:
        keys, values = keys_new, values_new

    # ...

    num_tokens_Q = queries.shape[-2]
    num_tokens_K = keys.shape[-2]
    if use_cache:
        mask_bool = self.mask.bool()[
            self.ptr_current_pos:self.ptr_current_pos + num_tokens_Q, :num_tokens_K
        ]
        self.ptr_current_pos += num_tokens_Q
    else:
        mask_bool = self.mask.bool()[:num_tokens_Q, :num_tokens_K]

In [ ]:
"""
When generating texts, between independent sequences (for instance to text generation calls) we must reset both buffers, so we also add a cache resetting method the to the MultiHeadAttention class
"""

def reset_cache(self):
    self.cache_k, self.cache_v = None, None
    self.ptr_current_pos = 0

In [ ]:
def generate_text_simple_cached(model, idx, max_new_tokens,
                                context_size=None, use_cache=True):
    model.eval()
    ctx_len = context_size or model.pos_emb.num_embeddings

    with torch.no_grad():
        if use_cache:
            # Init cache with full prompt
            model.reset_kv_cache()
            logits = model(idx[:, -ctx_len:], use_cache=True)

            for _ in range(max_new_tokens):
                # a) pick the token with the highest log-probability (greedy sampling)
                next_idx = logits[:, -1].argmax(dim=-1, keepdim=True)
                # b) append it to the running sequence
                idx = torch.cat([idx, next_idx], dim=1)
                # c) feed model only the new token
                logits = model(next_idx, use_cache=True)
        else:
            for _ in range(max_new_tokens):
                logits = model(idx[:, -ctx_len:], use_cache=False)
                next_idx = logits[:, -1].argmax(dim=-1, keepdim=True)
                idx = torch.cat([idx, next_idx], dim=1)

    return idx

# Weight Tying

### Definition

**Weight Tying** || Input Embedding Matrix and Output Projection Matrix share the same weights.

```text
W_out = W_E.T
```

Instead of storing separate `W_E` and `W_out`, GPT reuses the embedding matrix.

---

### Where They Are Used

**Input Side**

Token IDs:

```text
[10,25,42]
```

Embedding Matrix:

```text
W_E : (50257,768)
```

Operation:

```text
Token ID
↓
Lookup Row from W_E
↓
768-d Vector
```

Converts:

```text
Token → Meaning Vector
```

---

**Output Side**

Final Hidden States:

```text
(B,T,768)
```

Need Vocabulary Scores:

```text
(B,T,50257)
```

Operation:

```text
logits = hidden_states @ W_out
```

Without weight tying:

```text
W_out : (768,50257)
```

---

### Without Weight Tying

Separate matrices:

```text
W_E   : (50257,768)

W_out : (768,50257)
```

Parameter Count:

```text
50257×768
+
768×50257

≈ 77M
```

parameters just for embeddings and output layer.

---

### With Weight Tying

Reuse embedding matrix:

```text
W_out = W_E.T
```

Shape:

```text
W_E    : (50257,768)

W_E.T  : (768,50257)
```

Output:

```text
logits = hidden_states @ W_E.T
```

Parameter Count:

```text
50257×768

≈ 38M
```

Almost 50% fewer parameters.

---

### Intuition

Embedding Layer:

```text
Token
↓
Meaning Vector
```

Example:

```text
cat
↓
[0.2,0.7,-0.1,...]
```

Output Projection:

```text
Meaning Vector
↓
Token
```

Example:

```text
[0.2,0.7,-0.1,...]
↓
cat
```

**Key Idea** || Embeddings map **Token → Vector** while output projection maps **Vector → Token**, making them natural inverse operations.

---

### How Prediction Works

Final Hidden State:

```text
h : (768,)
```

Compute:

```text
h @ W_E.T
```

Output:

```text
(50257,)
```

One score per vocabulary token.

Interpretation:

```text
How similar is h
to each token embedding?
```


---

### Why It Works

Hidden state `h` lives in the same embedding space as token embeddings.

Output layer effectively asks:

```text
Which token embedding
is most similar to h?
```

The token with the highest similarity gets the highest probability.

---

### Training

Backprop updates:

```text
W_E
```

only once.

Since:

```text
W_out = W_E.T
```

the same update improves:

* Input embeddings
* Output predictions

simultaneously.



In [ ]:
self.tok_emb = nn.Embedding(
    vocab_size,
    emb_dim
)

self.out_head = nn.Linear(
    emb_dim,
    vocab_size,
    bias=False
)

self.out_head.weight = self.tok_emb.weight

# Temperature and Top-K Sampling

### Purpose

After GPT produces logits:

```text
(50257,)
```

we must choose the next token.

**Problem with Greedy (`argmax`)** || Always selects the highest-probability token || Deterministic, repetitive, and less creative output.

---

## Temperature Sampling

### Definition

Controls how random the generation should be.

Formula:

```text
Softmax(logits / T)
```

---

### T = 1

Normal softmax.

```text
cat  = 0.86
dog  = 0.12
car  = 0.015
tree = 0.005
```

No change.

---

### T < 1

Example:

```text
T = 0.5
```

Effect || Logit differences become larger || Distribution becomes sharper.

```text
cat  = 0.98
dog  = 0.02
others ≈ 0
```

Result:

* More confident
* More deterministic
* Less creative

---

### T > 1

Example:

```text
T = 2
```

Effect || Logit differences become smaller || Distribution becomes flatter.

```text
cat  = 0.60
dog  = 0.25
car  = 0.10
tree = 0.05
```

Result:

* More randomness
* More diversity
* More creativity

---

### Extreme Cases

**T → 0** || Approaches `argmax()` || Always picks the best token.

**Very Large T** || Probabilities become nearly equal || Output becomes almost random.

---

## Top-K Sampling

### Problem

High temperature can allow very unlikely tokens to be selected.

Example:

```text
cat     50%
dog     30%
car     15%
tree     4%
banana 0.5%
rocket 0.1%
```

GPT might sample:

```text
rocket
```

even though it makes little sense.

---

### Definition

Keep only the K highest-probability tokens and discard the rest.

Example:

```text
K = 3
```

Keep:

```text
cat
dog
car
```

Remove:

```text
tree
banana
rocket
```

---

### After Top-K

Before:

```text
cat     50%
dog     30%
car     15%
tree     4%
banana 0.5%
rocket 0.1%
```

After filtering and renormalizing:

```text
cat = 52.6%
dog = 31.6%
car = 15.8%
others = 0%
```

Then sample from the remaining tokens.

---

### Extreme Cases

**K = 1** || Equivalent to `argmax()`.

**K = Vocabulary Size** || Equivalent to normal sampling.

---

## Why Use Both?

Temperature controls **randomness of probabilities**.

Top-K controls **which tokens are allowed to participate**.

Typical pipeline:

```text
Logits
↓
Temperature
↓
Top-K
↓
Softmax
↓
Sampling
```

Common settings:

```text
temperature = 0.8
top_k = 50
```

---

### One-Line Summary

**Temperature** || Controls *how random* token probabilities are.

**Top-K** || Restricts sampling to the *K most likely tokens*, preventing unlikely/nonsensical choices.


In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)

new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float("-inf")),
    other=next_token_logits
)


In [ ]:
# This file collects all the relevant code that we covered thus far
# throughout Chapters 3-4.
# This file can be run as a standalone script.

import time
import tiktoken
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
            persistent=False
        )

        ####################################################
        # NEW
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.ptr_current_pos = 0
        ####################################################

    def forward(self, x, use_cache=False):
        b, num_tokens, d_in = x.shape

        keys_new = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        values_new = self.W_value(x)
        queries = self.W_query(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys_new = keys_new.view(b, num_tokens, self.num_heads, self.head_dim)
        values_new = values_new.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        ####################################################
        # NEW
        if use_cache:
            if self.cache_k is None:
                self.cache_k, self.cache_v = keys_new, values_new
            else:
                self.cache_k = torch.cat([self.cache_k, keys_new], dim=1)
                self.cache_v = torch.cat([self.cache_v, values_new], dim=1)
            keys, values = self.cache_k, self.cache_v
        else:
            keys, values = keys_new, values_new
        ####################################################

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        ####################################################
        # NEW
        num_tokens_Q = queries.shape[-2]
        num_tokens_K = keys.shape[-2]
        if use_cache:
            mask_bool = self.mask.bool()[
                self.ptr_current_pos:self.ptr_current_pos + num_tokens_Q, :num_tokens_K
            ]
            self.ptr_current_pos += num_tokens_Q
        ####################################################
        # Original mask truncated to the number of tokens and converted to boolean
        else:
            mask_bool = self.mask.bool()[:num_tokens_Q, :num_tokens_K]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

    ####################################################
    # NEW
    def reset_cache(self):
        self.cache_k, self.cache_v = None, None
        self.ptr_current_pos = 0
    ####################################################


#####################################
# Chapter 4
#####################################
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, use_cache=False):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)

        # x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        ####################################################
        # NEW
        x = self.att(x, use_cache=use_cache)
        ####################################################

        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # self.trf_blocks = nn.Sequential(
        #    *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        ####################################################
        # NEW
        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.current_pos = 0
        ####################################################

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx, use_cache=False):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        # pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        ####################################################
        # NEW

        if use_cache:
            pos_ids = torch.arange(self.current_pos, self.current_pos + seq_len, device=in_idx.device, dtype=torch.long)
            self.current_pos += seq_len
        else:
            pos_ids = torch.arange(0, seq_len, device=in_idx.device, dtype=torch.long)
        pos_embeds = self.pos_emb(pos_ids).unsqueeze(0)
        ####################################################

        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)

        # x = self.trf_blocks(x)
        ####################################################
        # NEW
        for blk in self.trf_blocks:
            x = blk(x, use_cache=use_cache)
        ####################################################

        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

    ####################################################
    # NEW
    def reset_kv_cache(self):
        for blk in self.trf_blocks:
            blk.att.reset_cache()
        self.current_pos = 0
    ####################################################


def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx


####################################################
# NEW
def generate_text_simple_cached(model, idx, max_new_tokens,
                                context_size=None, use_cache=True):
    model.eval()
    ctx_len = context_size or model.pos_emb.num_embeddings

    with torch.no_grad():
        if use_cache:
            # Init cache with full prompt
            model.reset_kv_cache()
            logits = model(idx[:, -ctx_len:], use_cache=True)

            for _ in range(max_new_tokens):
                # a) pick the token with the highest log-probability (greedy sampling)
                next_idx = logits[:, -1].argmax(dim=-1, keepdim=True)
                # b) append it to the running sequence
                idx = torch.cat([idx, next_idx], dim=1)
                # c) feed model only the new token
                logits = model(next_idx, use_cache=True)
        else:
            for _ in range(max_new_tokens):
                logits = model(idx[:, -ctx_len:], use_cache=False)
                next_idx = logits[:, -1].argmax(dim=-1, keepdim=True)
                idx = torch.cat([idx, next_idx], dim=1)

    return idx
####################################################


def main():
    GPT_CONFIG_124M = {
        "vocab_size": 50257,     # Vocabulary size
        "context_length": 1024,  # Context length
        "emb_dim": 768,          # Embedding dimension
        "n_heads": 12,           # Number of attention heads
        "n_layers": 12,          # Number of layers
        "drop_rate": 0.1,        # Dropout rate
        "qkv_bias": False        # Query-Key-Value bias
    }

    torch.manual_seed(123)
    model = GPTModel(GPT_CONFIG_124M)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # disable dropout

    start_context = "Hello, I am"

    tokenizer = tiktoken.get_encoding("gpt2")
    encoded = tokenizer.encode(start_context)
    encoded_tensor = torch.tensor(encoded, device=device).unsqueeze(0)

    print(f"\n{50*'='}\n{22*' '}IN\n{50*'='}")
    print("\nInput text:", start_context)
    print("Encoded input text:", encoded)
    print("encoded_tensor.shape:", encoded_tensor.shape)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()

    # token_ids = generate_text_simple(
    #     model=model,
    #     idx=encoded_tensor,
    #     max_new_tokens=200,
    #     context_size=GPT_CONFIG_124M["context_length"]
    # )

    ####################################################
    # NEW
    token_ids = generate_text_simple_cached(
        model=model,
        idx=encoded_tensor,
        max_new_tokens=200,
    )
    ####################################################

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    total_time = time.time() - start

    decoded_text = tokenizer.decode(token_ids.squeeze(0).tolist())

    print(f"\n\n{50*'='}\n{22*' '}OUT\n{50*'='}")
    print("\nOutput:", token_ids)
    print("Output length:", len(token_ids[0]))
    print("Output text:", decoded_text)

    print(f"\nTime: {total_time:.2f} sec")
    print(f"{int(len(token_ids[0])/total_time)} tokens/sec")
    if torch.cuda.is_available():
        max_mem_bytes = torch.cuda.max_memory_allocated()
        max_mem_gb = max_mem_bytes / (1024 ** 3)
        print(f"Max memory allocated: {max_mem_gb:.2f} GB")


if __name__ == "__main__":
    main()